# **Modelo LightFM - Base**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

In [1]:
# !pip install git+https://github.com/daviddavo/lightfm

In [2]:
import os
import sys
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightfm
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k as lightfm_prec_at_k
from lightfm.evaluation import recall_at_k as lightfm_recall_at_k

print("System version: {}".format(sys.version))
print("LightFM version: {}".format(lightfm.__version__))


System version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
LightFM version: 1.17


In [3]:
DATA_SIZE = 'small'

K = 10
TEST_PERCENTAGE = 0.25
LEARNING_RATE = 0.1
LOSS_FUNCTION = 'bpr'
NO_COMPONENTS = 20
NO_EPOCHS = 20
NO_THREADS = 32
ITEM_ALPHA = 1e-6
USER_ALPHA = 1e-6

SEED = 42

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from lightfm.data import Dataset

Ahora la idea es hacer un preprocesamiento básico, para que después no tengamos problemas

In [5]:
DATA_PATH = "video_game_reviews_with_userid_clean.csv"
data = pd.read_csv(DATA_PATH)
data = data.rename(columns={
    "user_id": "userID",
    "item_id": "itemID",
    "rating": "rating"
})

data = data.dropna(subset=["userID", "itemID", "rating"]).copy()
data["userID"] = data["userID"].astype(str)
data["itemID"] = data["itemID"].astype(str)
data["rating"] = pd.to_numeric(data["rating"], errors="coerce")

In [6]:
print(f"datos cargados: {len(data)} filas, {data['userID'].nunique()} usuarios, {data['itemID'].nunique()} ítems")
display(data.sample(5, random_state=SEED))

datos cargados: 47774 filas, 3000 usuarios, 40 ítems


,userID,itemID,rating
20481,1621,38,2.116751
21278,531,4,3.609137
37809,516,18,2.736041
34452,2731,3,3.436548
35793,2648,14,3.172589


Todo en orden, ahora vamos a definir una función para garantizar un split que le permita a LightFM trabajar sin problemas. La idea es que no haya intersección alguna entre el test y el train.

In [7]:
def stratified_user_split(df, test_size=0.25, seed=42):
    train_parts, test_parts = [], []
    for user, grp in df.groupby("userID"):
        if len(grp) < 2:
            train_parts.append(grp)
            continue
        tr, te = train_test_split(grp, test_size=test_size, random_state=seed)
        train_parts.append(tr)
        test_parts.append(te)
    return pd.concat(train_parts), pd.concat(test_parts)


Nos vamos a aseguraer de que no hayan duplicados, o bien si es que hay, los borraremos

In [ ]:
print(f"Antes de eliminar duplicados: {data.shape}")
data = data.drop_duplicates(subset=["userID", "itemID"], keep="first")
print(f"Después de eliminar duplicados: {data.shape}")


train_df, test_df = stratified_user_split(data, test_size=TEST_PERCENTAGE, seed=SEED)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Antes de eliminar duplicados: (47774, 3)
Después de eliminar duplicados: (39441, 3)
Train: (28451, 3), Test: (10990, 3)


Ahora viene la parte de construir el dataset y las matrices, muy en la línea de lo visto en el práctico

In [9]:
dataset = Dataset()
dataset.fit(
    users=data["userID"].unique(),
    items=data["itemID"].unique()
)

(interactions, weights) = dataset.build_interactions(data[["userID", "itemID", "rating"]].values)
(train_interactions, train_weights) = dataset.build_interactions(train_df[["userID", "itemID", "rating"]].values)
(test_interactions, test_weights) = dataset.build_interactions(test_df[["userID", "itemID", "rating"]].values)

print(f"interacciones totales: {interactions.shape}")
print(f"train: {train_interactions.shape}, test: {test_interactions.shape}")


interacciones totales: (3000, 40)
train: (3000, 40), test: (3000, 40)


In [10]:
print(f"total interacciones: {interactions.nnz}")
print(f"train interacciones: {train_interactions.nnz}")
print(f"test interacciones: {test_interactions.nnz}")

total interacciones: 39441
train interacciones: 28451
test interacciones: 10990


In [11]:
model1 = LightFM(
    loss=LOSS_FUNCTION,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=SEED
)

model1.fit(train_interactions, epochs=NO_EPOCHS, num_threads=NO_THREADS)


In [12]:
eval_precision_lfm = lightfm_prec_at_k(
    model1,
    test_interactions,
    train_interactions=train_interactions,
    k=K
).mean()

eval_recall_lfm = lightfm_recall_at_k(
    model1,
    test_interactions,
    train_interactions=train_interactions,
    k=K
).mean()

print(f"precision@{K}: {eval_precision_lfm:.4f}")
print(f"recall@{K}:    {eval_recall_lfm:.4f}")



precision@10: 0.1208
recall@10:    0.3257


Ahora viene la parte de hacer una validación cruzada. Trataremos de buscar la mejor combinación posible de parámetros. Aprovecho de dejar una referencia que nos sirvió.

- https://pypi.org/project/tqdm/

In [ ]:
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k
from tqdm import tqdm
import numpy as np
import pandas as pd

def evaluate_lightfm_models(
    train_interactions,
    test_interactions,
    param_grid,
    K=10,
    num_threads=8,
    max_epochs=20,
    random_state=42
):
    # basicamente evalua múltiples configuraciones de LightFM sobre un único split
    # (usando las matrices train_interactions y test_interactions ya construidas)
    results = []

    for params in tqdm(param_grid, desc="Evaluando hiperparámetros"):
        # modelo
        model = LightFM(
            loss=params['loss'],
            no_components=params['no_components'],
            learning_rate=params['learning_rate'],
            random_state=np.random.RandomState(random_state)
        )

        # entrenamiento
        model.fit(train_interactions, epochs=max_epochs, num_threads=num_threads)

        # evaluación
        prec = precision_at_k(
            model, test_interactions,
            train_interactions=train_interactions,
            k=K, num_threads=num_threads
        ).mean()

        rec = recall_at_k(
            model, test_interactions,
            train_interactions=train_interactions,
            k=K, num_threads=num_threads
        ).mean()

        results.append({
            'params': params,
            f'precision@{K}': prec,
            f'recall@{K}': rec
        })

    return pd.DataFrame(results)


In [14]:
param_grid = [
    {'loss': 'warp', 'no_components': 16, 'learning_rate': 0.05},
    {'loss': 'warp', 'no_components': 32, 'learning_rate': 0.05},
    {'loss': 'warp', 'no_components': 32, 'learning_rate': 0.1},
    {'loss': 'bpr',  'no_components': 32, 'learning_rate': 0.05},
    {'loss': 'bpr',  'no_components': 64, 'learning_rate': 0.05},
]

results = evaluate_lightfm_models(
    train_interactions=train_interactions,
    test_interactions=test_interactions,
    param_grid=param_grid,
    K=10,
    num_threads=8,
    max_epochs=20,
    random_state=SEED
)

print(results.sort_values(by='precision@10', ascending=False))


Evaluando hiperparámetros: 100%|██████████| 5/5 [00:44<00:00,  8.95s/it]

                                              params  precision@10  recall@10
1  {'loss': 'warp', 'no_components': 32, 'learnin...      0.123267   0.330561
0  {'loss': 'warp', 'no_components': 16, 'learnin...      0.122200   0.330483
3  {'loss': 'bpr', 'no_components': 32, 'learning...      0.121467   0.327767
4  {'loss': 'bpr', 'no_components': 64, 'learning...      0.121267   0.326039
2  {'loss': 'warp', 'no_components': 32, 'learnin...      0.121000   0.326739


Ahora ya tenemos la mejor selección de parámetros de esta validación.

In [15]:
LOSS_FUNCTION = 'warp'
NO_COMPONENTS = 32
LEARNING_RATE = 0.1
ITEM_ALPHA = 1e-6
USER_ALPHA = 1e-6


In [16]:
model_final = LightFM(
    loss=LOSS_FUNCTION,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=SEED
)

model_final.fit(train_interactions, epochs=20, num_threads=8)


In [17]:
eval_precision_lfm = lightfm_prec_at_k(
    model_final,
    test_interactions,
    train_interactions=train_interactions,
    k=K
).mean()

eval_recall_lfm = lightfm_recall_at_k(
    model_final,
    test_interactions,
    train_interactions=train_interactions,
    k=K
).mean()

print(f"Precision@{K}: {eval_precision_lfm:.4f}")
print(f"Recall@{K}:    {eval_recall_lfm:.4f}")



Precision@10: 0.1212
Recall@10:    0.3260


Recordemos que hay cierta aleatoriedad intrínseca en LightFM, incluso replicando la misma semilla, así que el resultado es el esperado.

## Métricas

La idea de este apartado es poder adaptar el dataset para que se puedan calcular las métricas que hemos definido. Así, vamos a trabajar con un modo específico que replicaremos en los demás cuadernos.

In [18]:
ratings = pd.read_csv("video_game_reviews_with_userid_clean.csv")
metadata = pd.read_csv("video_game_reviews.csv")

ratings = ratings.rename(columns={
    "user_id": "userID",
    "item_id": "itemID",
    "rating": "rating"
})

ratings["userID"] = ratings["userID"].astype(str)
ratings["itemID"] = ratings["itemID"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

ratings = ratings.drop_duplicates(subset=["userID", "itemID"])
print(f"Ratings: {ratings.shape}")

Ratings: (39441, 3)


In [19]:
# metadata con nombres de juegos
metadata = metadata.rename(columns={"Game Title": "GameTitle"})
metadata["GameTitle"] = metadata["GameTitle"].astype(str).str.strip()
print(f"Metadata: {metadata.shape}")

feature_cols = [
    "Genre", "Platform", "Developer", "Publisher",
    "Game Mode", "Multiplayer", "Requires Special Device",
    "Age Group Targeted"
]

def make_features(row):
    feats = []
    for c in feature_cols:
        if c in row and pd.notna(row[c]):
            feats.append(f"{c}={str(row[c]).strip()}")
    return feats

metadata["features"] = metadata.apply(make_features, axis=1)

Metadata: (47774, 18)


In [20]:
unique_items = sorted(ratings["itemID"].unique())
metadata = metadata.head(len(unique_items)).copy()
metadata["itemID"] = unique_items[:len(metadata)]
metadata["itemID"] = metadata["itemID"].astype(str)

In [ ]:
# mapeos para convertir entre índices internos y ids originales
user_id_map, user_feat_map, item_id_map, item_feat_map = dataset.mapping()
inv_item_id_map = {v: k for k, v in item_id_map.items()}
inv_user_id_map = {v: k for k, v in user_id_map.items()}

# recomendar para un usuario (por indice interno de LightFM)
def recommend_for_user(model, user_internal_id, train_interactions, K=10,
                       user_features=None, item_features=None):
    n_users, n_items = train_interactions.shape

    # predice para tds los items
    item_ids = np.arange(n_items)
    scores = model.predict(
        user_ids=user_internal_id,
        item_ids=item_ids,
        user_features=user_features,
        item_features=item_features
    )

    # "enmascara" items ya vistos por ese usuario en el set de training
    known_items = train_interactions.tocsr()[user_internal_id].indices
    scores[known_items] = -np.inf
    valid_mask = np.isfinite(scores)
    num_candidates = int(valid_mask.sum())

    # Top-K, 10 es lo que usamos pero igual es mejor dejarlo parametrizado por si acaso
    K_eff = min(K, num_candidates) if num_candidates > 0 else 0
    top_items = np.argsort(-scores)[:K_eff]

    # convierte indices internos a ids originales
    rec_item_ids = [inv_item_id_map[i] for i in top_items]

    return {
        "requested_K": K,
        "available_candidates": num_candidates,
        "returned_K": K_eff,
        "internal_item_indices": top_items.tolist(),
        "item_ids": rec_item_ids
    }

# como ejemplo, la idea es elegir un usuario válido, por índice interno
example_user_internal_id = 2

res = recommend_for_user(
    model=model_final,
    user_internal_id=example_user_internal_id,
    train_interactions=train_interactions,
    K=K
)

print(f"usuario interno: {example_user_internal_id} (ID original: {inv_user_id_map.get(example_user_internal_id)})")
print(f"candidatos disponibles (no vistos): {res['available_candidates']}")
print(f"solicitados K={res['requested_K']}, Devueltos: {res['returned_K']}")
print("recomendaciones (itemID):")
for iid in res["item_ids"]:
    print("  -", iid)


usuario interno: 2 (ID original: 1131)
candidatos disponibles (no vistos): 31
solicitados K=10, Devueltos: 10
recomendaciones (itemID):
  - 17
  - 2
  - 19
  - 5
  - 7
  - 21
  - 27
  - 39
  - 3
  - 20


In [22]:
rec_df = pd.DataFrame({"itemID": res["item_ids"]})
rec_df = rec_df.merge(metadata[["itemID", "GameTitle"]], on="itemID", how="left")

print("recomendaciones para el usuario 1131:")
display(rec_df)


recomendaciones para el usuario 1131:


,itemID,GameTitle
0,17,Sid Meier’s Civilization VI
1,2,Street Fighter V
2,19,Spelunky 2
3,5,Sid Meier’s Civilization VI
4,7,Spelunky 2
5,21,Fall Guys
6,27,Stardew Valley
7,39,Fortnite
8,3,The Legend of Zelda: Breath of the Wild
9,20,Street Fighter V


Ahora ya tenemos las métricas que hemos usado. Notemos que igual definimos precisión y recall, que ya vienen por defecto. La idea es ser consistente con los modelos de referencia, así que por eso los pusimos aquí.

In [23]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

Finalmente ya podemos hacer un mapeo de id item a titulo y genero, y así ya podemos evaluar el modelo en su conjunto.

In [ ]:
itemid2title = dict(metadata[["itemID", "GameTitle"]].drop_duplicates("itemID").values)
itemid2genre = dict(metadata[["itemID", "Genre"]].drop_duplicates("itemID").values)

info_videojuegos = {}
for internal_idx, original_item_id in inv_item_id_map.items():
    title = itemid2title.get(str(original_item_id))
    genre = itemid2genre.get(str(original_item_id))
    info_videojuegos[internal_idx] = (title, genre)

def evaluar_metricas(
    model,
    train_interactions,
    test_interactions,
    K=10,
    num_threads=NO_THREADS,
    info_videojuegos=None
):
    n_users, n_items = train_interactions.shape
    train_csr = train_interactions.tocsr()
    test_csr  = test_interactions.tocsr()

    rec_k_dict = {}
    rows_user = []

    for u in range(n_users):
        # relevantes en test para el usuario
        rel_set = set(test_csr[u].indices)
        item_ids = np.arange(n_items)

        scores = model.predict(
            user_ids=u,
            item_ids=item_ids,
            num_threads=num_threads
        )

        # enmascarar items ya vistos en train
        scores[train_csr[u].indices] = -np.inf

        valid = np.isfinite(scores)
        if not valid.any():
            rec_k = []
        else:
            K_eff = min(K, int(valid.sum()))
            top_idx = np.argsort(-scores)[:K_eff]
            rec_k = list(top_idx)

        rec_k_dict[u] = rec_k

        prec = precision_at_k(rec_k, rel_set)
        rec  = recall_at_k(rec_k, rel_set)
        ndcg = ndcg_at_k(rec_k, rel_set)
        hit  = hit_score_at_k(rec_k, rel_set)
        m_ap = map_at_k(rec_k, rel_set)

        rows_user.append({
            "user_internal": u,
            "relevantes_test": len(rel_set),
            "K_devueltos": len(rec_k),
            "precision@K": prec,
            "recall@K": rec,
            "ndcg@K": ndcg,
            "hit_score@K": hit,
            "map@K": m_ap
        })

    df_users = pd.DataFrame(rows_user)
    global_metrics = {
        "users_evaluated": int(df_users.shape[0]),
        "precision@K": float(df_users["precision@K"].mean()) if not df_users.empty else np.nan,
        "recall@K": float(df_users["recall@K"].mean()) if not df_users.empty else np.nan,
        "ndcg@K": float(df_users["ndcg@K"].mean()) if not df_users.empty else np.nan,
        "hit_score@K": float(df_users["hit_score@K"].mean()) if not df_users.empty else np.nan,
        "map@K": float(df_users["map@K"].mean()) if not df_users.empty else np.nan
    }

    if info_videojuegos is not None and len(info_videojuegos) > 0:
        global_metrics["diversity@K"] = float(diversity_at_k(rec_k_dict, info_videojuegos))
    else:
        global_metrics["diversity@K"] = np.nan

    df_global = pd.DataFrame([global_metrics])

    return df_global

df_global = evaluar_metricas(
    model=model_final,
    train_interactions=train_interactions,
    test_interactions=test_interactions,
    K=K,
    num_threads=NO_THREADS,
    info_videojuegos=info_videojuegos
)

display(df_global)

,users_evaluated,precision@K,recall@K,ndcg@K,hit_score@K,map@K,diversity@K
0,3000,0.1212,0.325989,0.224797,0.773333,0.117378,6.834


Referencias adicionales
- https://www.stepbystepdatascience.com/hybrid-recommender-lightfm-python